# ============================================================
# SECTION 1: SETUP & PREPROCESSING
# ============================================================

In [11]:
# ============================================================
# Jalankan file ini pertama kali sebelum file lainnya
# ============================================================

import numpy as np
import pandas as pd
import json
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [12]:
# ------------------------------------------------------------
# 1.1 Load Dataset & Load data ECOCROP:
# ------------------------------------------------------------
df = pd.read_csv('../dataset/Crop_recommendation.csv')
print("Shape dataset:", df.shape)
print(df.head())
print("\nJumlah sampel per kelas:")
print(df['label'].value_counts())

with open("../dataset/constraints_optimal.json", "r") as f:
    ecocrop = json.load(f)
print("\nContoh data ECOCROP (rice):", ecocrop['rice'])

Shape dataset: (2200, 8)
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice

Jumlah sampel per kelas:
label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64

Contoh data ECOCROP (rice): {'temperature': [20, 30], 'ph': [5.5, 7.0]}


In [13]:
# ------------------------------------------------------------
# 1.2 Label Encoding
# ------------------------------------------------------------
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\nLabel mapping:", label_mapping)


Label mapping: {'apple': np.int64(0), 'banana': np.int64(1), 'blackgram': np.int64(2), 'chickpea': np.int64(3), 'coconut': np.int64(4), 'coffee': np.int64(5), 'cotton': np.int64(6), 'grapes': np.int64(7), 'jute': np.int64(8), 'kidneybeans': np.int64(9), 'lentil': np.int64(10), 'maize': np.int64(11), 'mango': np.int64(12), 'mothbeans': np.int64(13), 'mungbean': np.int64(14), 'muskmelon': np.int64(15), 'orange': np.int64(16), 'papaya': np.int64(17), 'pigeonpeas': np.int64(18), 'pomegranate': np.int64(19), 'rice': np.int64(20), 'watermelon': np.int64(21)}


In [14]:
# ------------------------------------------------------------
# 1.3 Pisahkan Fitur dan Target
# ------------------------------------------------------------
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].values
y = df['label_encoded'].values

print("\nShape X:", X.shape)
print("Shape y:", y.shape)


Shape X: (2200, 7)
Shape y: (2200,)


In [15]:
# ------------------------------------------------------------
# 1.4 Split Data (70:15:15 stratified)
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\nTrain:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)



Train: (1540, 7)
Val  : (330, 7)
Test : (330, 7)


In [16]:
# ------------------------------------------------------------
# 1.5 Feature Scaling (fit HANYA dari training set)
# ------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("\nMean tiap fitur (harusnya ~0):", X_train_scaled.mean(axis=0).round(3))
print("Std tiap fitur (harusnya ~1) :", X_train_scaled.std(axis=0).round(3))


Mean tiap fitur (harusnya ~0): [ 0.  0. -0. -0. -0. -0. -0.]
Std tiap fitur (harusnya ~1) : [1. 1. 1. 1. 1. 1. 1.]


In [17]:
# ------------------------------------------------------------
# 1.6 One-hot Encoding Label
# ------------------------------------------------------------
num_classes = 22
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  num_classes)

print("\nShape y_train_oh:", y_train_oh.shape)

print("\n[SECTION 1 SELESAI] Semua variabel preprocessing siap digunakan.")


Shape y_train_oh: (1540, 22)

[SECTION 1 SELESAI] Semua variabel preprocessing siap digunakan.


In [18]:
import pickle

saved_preprocessing = {
    'le':             le,
    'scaler':         scaler,
    'ecocrop':        ecocrop,
    'X_train_scaled': X_train_scaled,
    'X_val_scaled':   X_val_scaled,
    'X_test_scaled':  X_test_scaled,
    'X_train':        X_train,
    'X_val':          X_val,
    'X_test':         X_test,
    'y_train':        y_train,
    'y_val':          y_val,
    'y_test':         y_test,
    'y_train_oh':     y_train_oh,
    'y_val_oh':       y_val_oh,
    'y_test_oh':      y_test_oh,
}

with open('session_preprocessing.pkl', 'wb') as f:
    pickle.dump(saved_preprocessing, f)

print("✅ Preprocessing disimpan ke session_preprocessing.pkl")

✅ Preprocessing disimpan ke session_preprocessing.pkl
